# 🧹 Data Cleaning Property Investment - Listings

At this stage, I have successfully scraped raw property listing data from multiple sources.

However, raw scraped data is **not usable for analysis or modeling** due to:

- Missing values
- Inconsistent formats (prices, suburbs, property types)
- Duplicates
- Incorrect data types
- Outliers and invalid values

My goal in this notebook is to:

1. Build a **reproducible cleaning pipeline**
2. Standardize the dataset for **EDA and ML modeling**
3. Add **validation checks** to prevent bad data from propagating
4. Export a **clean dataset** ready for analysis

This is one of the most critical steps in the project. poor data = poor model.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

## 1) Loading Raw Data

I load the raw dataset generated from the scraping pipeline.

In [6]:
raw_path = Path("data\raw\listings\privateproperty_sales_final.csv")

df = pd.read_csv(raw_path)

print("Raw shape:", df.shape)
df.head()

Raw shape: (996, 25)


,source_site,listing_id,listing_url,title,purchase_price,suburb,city,province,property_type,bedrooms,...,levies,rates_taxes,description,listing_date,scraped_timestamp,pp_transaction_slug,pp_province_slug,pp_metro_slug,pp_city_slug,pp_suburb_slug
0,privateproperty,T5322389,https://www.privateproperty.co.za/for-sale/gau...,3 Bedroom House in Halfway Gardens,2996000.0,Halfway Gardens,Midrand,Gauteng,House,3.0,...,2420.0,1751.0,"3 Bedroom House in Halfway Gardens, On Show by...",2 Dec 2025,2026-03-27T15:29:56.546614+00:00,for-sale,gauteng,johannesburg-metro,midrand,halfway-gardens
1,privateproperty,T5413400,https://www.privateproperty.co.za/for-sale/gau...,4 Bedroom House in Linden,2475000.0,Linden,Northcliff,Gauteng,House,4.0,...,NaN,1996.0,"4 Bedroom House in Linden, Loved and well-main...",5 Mar 2026,2026-03-27T15:29:59.301851+00:00,for-sale,gauteng,johannesburg-metro,northcliff,linden
2,privateproperty,T5399982,https://www.privateproperty.co.za/for-sale/gau...,3 Bedroom House in River Club,3999999.0,River Club,Sandton,Gauteng,House,3.0,...,NaN,2961.0,"3 Bedroom House in River Club, Sparkling and M...",23 Feb 2026,2026-03-27T15:30:02.255733+00:00,for-sale,gauteng,johannesburg-metro,sandton,river-club
3,privateproperty,T5419042,https://www.privateproperty.co.za/for-sale/gau...,4 Bedroom House in Parktown,4900000.0,Parktown,Rosebank And Parktown,Gauteng,House,4.0,...,NaN,4083.0,"4 Bedroom House in Parktown, Discover a home w...",10 Mar 2026,2026-03-27T15:30:05.293953+00:00,for-sale,gauteng,johannesburg-metro,rosebank-and-parktown,parktown
4,privateproperty,T5403283,https://www.privateproperty.co.za/for-sale/gau...,4 Bedroom House in Lyndhurst,2699000.0,Lyndhurst,Johannesburg Central,Gauteng,House,4.0,...,NaN,NaN,"4 Bedroom House in Lyndhurst, Tucked away in a...",25 Feb 2026,2026-03-27T15:30:09.331485+00:00,for-sale,gauteng,johannesburg-metro,johannesburg-central,lyndhurst


## 1) Initial Data Audit

Before cleaning, I need to understand:

- Missing values
- Data types
- Obvious inconsistencies

In [21]:
df.info()
df.isnull().sum().sort_values(ascending=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 996 entries, 0 to 995
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   source_site          996 non-null    object 
 1   listing_id           994 non-null    object 
 2   listing_url          996 non-null    object 
 3   title                996 non-null    object 
 4   purchase_price       992 non-null    float64
 5   suburb               996 non-null    object 
 6   city                 996 non-null    object 
 7   province             996 non-null    object 
 8   property_type        993 non-null    object 
 9   bedrooms             996 non-null    float64
 10  bathrooms            996 non-null    float64
 11  parking_spaces       876 non-null    float64
 12  garage               423 non-null    float64
 13  floor_area_sqm       785 non-null    float64
 14  land_area_sqm        574 non-null    float64
 15  levies               599 non-null    flo

garage                 573
land_area_sqm          422
levies                 397
floor_area_sqm         211
rates_taxes            188
parking_spaces         120
purchase_price           4
property_type            3
listing_id               2
pp_city_slug             0
pp_metro_slug            0
pp_province_slug         0
pp_transaction_slug      0
scraped_timestamp        0
listing_date             0
description              0
source_site              0
bathrooms                0
bedrooms                 0
province                 0
city                     0
suburb                   0
title                    0
listing_url              0
pp_suburb_slug           0
dtype: int64

In [20]:
df.describe()

,purchase_price,bedrooms,bathrooms,parking_spaces,garage,floor_area_sqm,land_area_sqm,levies,rates_taxes
count,9.920000e+02,996.000000,996.000000,876.000000,423.000000,785.000000,574.000000,599.000000,808.000000
mean,2.627828e+07,2.689257,2.074297,2.846461,2.091017,166.468153,228.369338,2545.089065,1563.180160
std,3.745360e+07,1.464170,1.404766,2.511141,0.964716,184.058517,251.866095,2159.011808,1591.810995
min,1.500000e+06,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,7.499712e+06,2.000000,1.000000,1.000000,1.000000,62.000000,4.000000,1398.000000,560.000000
50%,1.250000e+07,2.000000,2.000000,2.000000,2.000000,88.000000,181.500000,2000.000000,944.000000
75%,2.881000e+07,3.000000,2.000000,3.000000,2.000000,189.000000,317.000000,2900.000000,1972.750000
max,3.000000e+08,16.000000,12.000000,18.000000,6.000000,980.000000,996.000000,16000.000000,10828.000000


## 2) Standardizing Column Names

Ensure consistent naming across datasets (important for joins later).

In [8]:
df.columns = (df.columns.str.strip().str.lower().str.replace(" ", "_"))

df.columns.tolist()

['source_site',
 'listing_id',
 'listing_url',
 'title',
 'purchase_price',
 'suburb',
 'city',
 'province',
 'property_type',
 'bedrooms',
 'bathrooms',
 'parking_spaces',
 'garage',
 'floor_area_sqm',
 'land_area_sqm',
 'levies',
 'rates_taxes',
 'description',
 'listing_date',
 'scraped_timestamp',
 'pp_transaction_slug',
 'pp_province_slug',
 'pp_metro_slug',
 'pp_city_slug',
 'pp_suburb_slug']

## 3) Cleaning Price Column

Prices often come with:
- Currency symbols (R)
- Spaces / commas

I convert everything to numeric.

In [14]:
def clean_purchase_price(value):
    if pd.isna(value):
        return np.nan

    # If it's already a number → return safely
    if isinstance(value, (int, float)):
        return float(value)

    value = str(value).lower().strip()

    # Handle "million"
    if "million" in value:
        num = re.findall(r"[\d\.]+", value)
        return float(num[0]) * 1_000_000 if num else np.nan

    # Handle "k"
    if "k" in value:
        num = re.findall(r"[\d\.]+", value)
        return float(num[0]) * 1_000 if num else np.nan

    # Keep digits AND decimal point
    value = re.sub(r"[^\d\.]", "", value)

    return float(value) if value != "" else np.nan


# Apply
df["purchase_price"] = df["purchase_price"].apply(clean_purchase_price)

In [15]:
df["purchase_price"].sort_values(ascending=False).head(10)

106    300000000.0
201    300000000.0
247    299990000.0
200    255000000.0
413    240000000.0
57     225000000.0
329    200000000.0
104    197500000.0
105    197500000.0
17     195000000.0
Name: purchase_price, dtype: float64

In [16]:
cols_to_check = [
    "purchase_price",
    "price",
    "title",
    "property_type",
    "suburb",
    "city",
    "province",
    "listing_url"
]

existing_cols = [c for c in cols_to_check if c in df.columns]

high_prices = df.loc[df["purchase_price"] >= 100_000_000, existing_cols]
high_prices

,purchase_price,title,property_type,suburb,city,province,listing_url
17,195000000.0,5 Bedroom House in Bryanston,House,Bryanston,Sandton,Gauteng,https://www.privateproperty.co.za/for-sale/gau...
25,105000000.0,5 Bedroom House in Bryanston,House,Bryanston,Sandton,Gauteng,https://www.privateproperty.co.za/for-sale/gau...
35,179990000.0,5 Bedroom House in Bryanston,House,Bryanston,Sandton,Gauteng,https://www.privateproperty.co.za/for-sale/gau...
56,180000000.0,5 Bedroom House in Blue Hills Equestrian Estate,House,Blue Hills,Midrand,Gauteng,https://www.privateproperty.co.za/for-sale/gau...
57,225000000.0,6 Bedroom House in Hyde Park,House,Hyde Park,Sandton,Gauteng,https://www.privateproperty.co.za/for-sale/gau...
103,109950000.0,5 Bedroom House in Fourways Gardens,House,Fourways Gardens,Sandton,Gauteng,https://www.privateproperty.co.za/for-sale/gau...
104,197500000.0,4 Bedroom House in Steyn City,House,Steyn City,Midrand,Gauteng,https://www.privateproperty.co.za/for-sale/gau...
105,197500000.0,4 Bedroom House in Melrose Estate,House,Melrose Estate,Rosebank And Parktown,Gauteng,https://www.privateproperty.co.za/for-sale/gau...
106,300000000.0,4 Bedroom House in Steyn City,House,Steyn City,Midrand,Gauteng,https://www.privateproperty.co.za/for-sale/gau...
190,115000000.0,4 Bedroom House in Helderfontein Estate,House,Helderfontein Estate,Midrand,Gauteng,https://www.privateproperty.co.za/for-sale/gau...


## 4) Cleaning Location Fields

Suburb data is often messy and inconsistent.

I standardize:
- Case formatting
- Remove whitespace

In [23]:
def clean_text(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().title()

for col in ["suburb", "city"]:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

## 5) Standardizing Property Types

Different sites use different labels:
- "Apartment"
- "Flat"
- "Townhouse"

I normalize these into consistent categories.

In [25]:
def normalize_property_type(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower()

    if "apartment" in x or "flat" in x:
        return "Apartment"
    elif "house" in x:
        return "House"
    elif "townhouse" in x:
        return "Townhouse"
    else:
        return "Other"

if "property_type" in df.columns:
    df["property_type"] = df["property_type"].apply(normalize_property_type)

## 6) Fixing Numeric Columns

Ensure correct types for modeling later.

In [26]:
numeric_cols = ["bedrooms", "bathrooms", "parking"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

## 7) Removing Duplicates

Duplicates can heavily bias analysis.

In [28]:
before = df.shape[0]

# Sort by most recent scrape
df = df.sort_values("scraped_timestamp", ascending=False)

# Remove duplicates by listing_id
df = df.drop_duplicates(subset=["listing_id"], keep="first")

after = df.shape[0]

print(f"Removed {before - after} duplicate rows")

Removed 1 duplicate rows


## 8) Handling Missing Values

Strategy:

- Drop rows missing **critical fields**
- Keep rows with minor missing values

In [33]:
critical_cols = ["purchase_price", "suburb", "city"]

df_clean = df.dropna(subset=critical_cols)

print("After dropping critical missing:", df_clean.shape)

After dropping critical missing: (991, 25)


In [34]:

# Columns to impute

group_cols = [c for c in ["suburb", "city", "province", "property_type"] if c in df_clean.columns]

# Make sure numeric columns are numeric
for col in ["rates_taxes", "levies"]:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Impute rates_taxes
# Rule:
#   a) fill missing values using median within same suburb/city/province/property_type
#   b) if still missing, use mean within same suburb/city/province/property_type
#   c) if still missing, use broader fallback means
# -----------------------------
if "rates_taxes" in df_clean.columns:
    # median at most detailed level
    rates_median_grp = df_clean.groupby(group_cols)["rates_taxes"].transform("median")
    df_clean["rates_taxes"] = df_clean["rates_taxes"].fillna(rates_median_grp)

    # mean at same detailed level
    rates_mean_grp = df_clean.groupby(group_cols)["rates_taxes"].transform("mean")
    df_clean["rates_taxes"] = df_clean["rates_taxes"].fillna(rates_mean_grp)

    # fallback 1: suburb + city + province
    fallback_cols_1 = [c for c in ["suburb", "city", "province"] if c in df_clean.columns]
    if fallback_cols_1:
        rates_mean_fallback_1 = df_clean.groupby(fallback_cols_1)["rates_taxes"].transform("mean")
        df_clean["rates_taxes"] = df_clean["rates_taxes"].fillna(rates_mean_fallback_1)

    # fallback 2: city + province
    fallback_cols_2 = [c for c in ["city", "province"] if c in df_clean.columns]
    if fallback_cols_2:
        rates_mean_fallback_2 = df_clean.groupby(fallback_cols_2)["rates_taxes"].transform("mean")
        df_clean["rates_taxes"] = df_clean["rates_taxes"].fillna(rates_mean_fallback_2)

    # fallback 3: province
    fallback_cols_3 = [c for c in ["province"] if c in df_clean.columns]
    if fallback_cols_3:
        rates_mean_fallback_3 = df_clean.groupby(fallback_cols_3)["rates_taxes"].transform("mean")
        df_clean["rates_taxes"] = df_clean["rates_taxes"].fillna(rates_mean_fallback_3)

    # final fallback: overall mean
    df_clean["rates_taxes"] = df_clean["rates_taxes"].fillna(df_clean["rates_taxes"].mean())

# Impute levies
# Rule:
#   use mean within same suburb/city/province/property_type
#   then broader fallbacks
# -----------------------------
if "levies" in df_clean.columns:
    # mean at most detailed level
    levies_mean_grp = df_clean.groupby(group_cols)["levies"].transform("mean")
    df_clean["levies"] = df_clean["levies"].fillna(levies_mean_grp)

    # fallback 1: suburb + city + province
    fallback_cols_1 = [c for c in ["suburb", "city", "province"] if c in df_clean.columns]
    if fallback_cols_1:
        levies_mean_fallback_1 = df_clean.groupby(fallback_cols_1)["levies"].transform("mean")
        df_clean["levies"] = df_clean["levies"].fillna(levies_mean_fallback_1)

    # fallback 2: city + province
    fallback_cols_2 = [c for c in ["city", "province"] if c in df_clean.columns]
    if fallback_cols_2:
        levies_mean_fallback_2 = df_clean.groupby(fallback_cols_2)["levies"].transform("mean")
        df_clean["levies"] = df_clean["levies"].fillna(levies_mean_fallback_2)

    # fallback 3: province
    fallback_cols_3 = [c for c in ["province"] if c in df_clean.columns]
    if fallback_cols_3:
        levies_mean_fallback_3 = df_clean.groupby(fallback_cols_3)["levies"].transform("mean")
        df_clean["levies"] = df_clean["levies"].fillna(levies_mean_fallback_3)

    # final fallback: overall mean
    df_clean["levies"] = df_clean["levies"].fillna(df_clean["levies"].mean())

# Check remaining missing values
# -----------------------------
check_cols = [c for c in ["price", "suburb", "city", "province", "property_type", "rates_taxes", "levies"] if c in df_clean.columns]

print("\nRemaining missing values:")
print(df_clean[check_cols].isna().sum())

print("\nFinal shape:")
print(df_clean.shape)


Remaining missing values:
suburb           0
city             0
province         0
property_type    3
rates_taxes      0
levies           0
dtype: int64

Final shape:
(991, 25)


C:\Users\ashle\AppData\Local\Temp\ipykernel_1748\1986971577.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
C:\Users\ashle\AppData\Local\Temp\ipykernel_1748\1986971577.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["rates_taxes"] = df_clean["rates_taxes"].fillna(rates_median_grp)
C:\Users\ashle\AppData\Local\Temp\ipykernel_1748\1986971577.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Da

In [36]:
df_clean.isnull().sum().sort_values(ascending=False)

garage                 569
land_area_sqm          419
floor_area_sqm         210
parking_spaces         118
property_type            3
listing_id               1
levies                   0
pp_city_slug             0
pp_metro_slug            0
pp_province_slug         0
pp_transaction_slug      0
scraped_timestamp        0
listing_date             0
description              0
rates_taxes              0
source_site              0
bathrooms                0
bedrooms                 0
province                 0
city                     0
suburb                   0
purchase_price           0
title                    0
listing_url              0
pp_suburb_slug           0
dtype: int64

In [38]:
df_clean.tail()

,source_site,listing_id,listing_url,title,purchase_price,suburb,city,province,property_type,bedrooms,...,levies,rates_taxes,description,listing_date,scraped_timestamp,pp_transaction_slug,pp_province_slug,pp_metro_slug,pp_city_slug,pp_suburb_slug
4,privateproperty,T5403283,https://www.privateproperty.co.za/for-sale/gau...,4 Bedroom House in Lyndhurst,26990000.0,Lyndhurst,Johannesburg Central,Gauteng,House,4.0,...,2130.000000,252.0,"4 Bedroom House in Lyndhurst, Tucked away in a...",25 Feb 2026,2026-03-27T15:30:09.331485+00:00,for-sale,gauteng,johannesburg-metro,johannesburg-central,lyndhurst
3,privateproperty,T5419042,https://www.privateproperty.co.za/for-sale/gau...,4 Bedroom House in Parktown,49000000.0,Parktown,Rosebank And Parktown,Gauteng,House,4.0,...,3082.336379,4083.0,"4 Bedroom House in Parktown, Discover a home w...",10 Mar 2026,2026-03-27T15:30:05.293953+00:00,for-sale,gauteng,johannesburg-metro,rosebank-and-parktown,parktown
2,privateproperty,T5399982,https://www.privateproperty.co.za/for-sale/gau...,3 Bedroom House in River Club,39999990.0,River Club,Sandton,Gauteng,House,3.0,...,5312.000000,2961.0,"3 Bedroom House in River Club, Sparkling and M...",23 Feb 2026,2026-03-27T15:30:02.255733+00:00,for-sale,gauteng,johannesburg-metro,sandton,river-club
1,privateproperty,T5413400,https://www.privateproperty.co.za/for-sale/gau...,4 Bedroom House in Linden,24750000.0,Linden,Northcliff,Gauteng,House,4.0,...,1389.000000,1996.0,"4 Bedroom House in Linden, Loved and well-main...",5 Mar 2026,2026-03-27T15:29:59.301851+00:00,for-sale,gauteng,johannesburg-metro,northcliff,linden
0,privateproperty,T5322389,https://www.privateproperty.co.za/for-sale/gau...,3 Bedroom House in Halfway Gardens,29960000.0,Halfway Gardens,Midrand,Gauteng,House,3.0,...,2420.000000,1751.0,"3 Bedroom House in Halfway Gardens, On Show by...",2 Dec 2025,2026-03-27T15:29:56.546614+00:00,for-sale,gauteng,johannesburg-metro,midrand,halfway-gardens


In [39]:
df_clean["purchase_price_Rands"] = df["purchase_price"] / 10
df_clean.tail()

C:\Users\ashle\AppData\Local\Temp\ipykernel_1748\1623958995.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["purchase_price_Rands"] = df["purchase_price"] / 10


,source_site,listing_id,listing_url,title,purchase_price,suburb,city,province,property_type,bedrooms,...,rates_taxes,description,listing_date,scraped_timestamp,pp_transaction_slug,pp_province_slug,pp_metro_slug,pp_city_slug,pp_suburb_slug,purchase_price_Rands
4,privateproperty,T5403283,https://www.privateproperty.co.za/for-sale/gau...,4 Bedroom House in Lyndhurst,26990000.0,Lyndhurst,Johannesburg Central,Gauteng,House,4.0,...,252.0,"4 Bedroom House in Lyndhurst, Tucked away in a...",25 Feb 2026,2026-03-27T15:30:09.331485+00:00,for-sale,gauteng,johannesburg-metro,johannesburg-central,lyndhurst,2699000.0
3,privateproperty,T5419042,https://www.privateproperty.co.za/for-sale/gau...,4 Bedroom House in Parktown,49000000.0,Parktown,Rosebank And Parktown,Gauteng,House,4.0,...,4083.0,"4 Bedroom House in Parktown, Discover a home w...",10 Mar 2026,2026-03-27T15:30:05.293953+00:00,for-sale,gauteng,johannesburg-metro,rosebank-and-parktown,parktown,4900000.0
2,privateproperty,T5399982,https://www.privateproperty.co.za/for-sale/gau...,3 Bedroom House in River Club,39999990.0,River Club,Sandton,Gauteng,House,3.0,...,2961.0,"3 Bedroom House in River Club, Sparkling and M...",23 Feb 2026,2026-03-27T15:30:02.255733+00:00,for-sale,gauteng,johannesburg-metro,sandton,river-club,3999999.0
1,privateproperty,T5413400,https://www.privateproperty.co.za/for-sale/gau...,4 Bedroom House in Linden,24750000.0,Linden,Northcliff,Gauteng,House,4.0,...,1996.0,"4 Bedroom House in Linden, Loved and well-main...",5 Mar 2026,2026-03-27T15:29:59.301851+00:00,for-sale,gauteng,johannesburg-metro,northcliff,linden,2475000.0
0,privateproperty,T5322389,https://www.privateproperty.co.za/for-sale/gau...,3 Bedroom House in Halfway Gardens,29960000.0,Halfway Gardens,Midrand,Gauteng,House,3.0,...,1751.0,"3 Bedroom House in Halfway Gardens, On Show by...",2 Dec 2025,2026-03-27T15:29:56.546614+00:00,for-sale,gauteng,johannesburg-metro,midrand,halfway-gardens,2996000.0


In [40]:
before = df_clean.shape[0]

# Clean property_type (handle empty strings)
df_clean["property_type"] = df_clean["property_type"].astype(str).str.strip()

# Remove rows with missing or invalid values
df_clean = df_clean[
    df_clean["listing_id"].notna() &
    df_clean["property_type"].notna() &
    (df_clean["property_type"] != "")
].copy()

after = df_clean.shape[0]

print(f"Removed {before - after} rows with missing listing_id or property_type")
print("New shape:", df_clean.shape)

Removed 1 rows with missing listing_id or property_type
New shape: (990, 26)


C:\Users\ashle\AppData\Local\Temp\ipykernel_1748\4109410517.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["property_type"] = df_clean["property_type"].astype(str).str.strip()


In [41]:
df_clean.isnull().sum().sort_values(ascending=False)

garage                  568
land_area_sqm           418
floor_area_sqm          210
parking_spaces          118
pp_suburb_slug            0
pp_city_slug              0
pp_metro_slug             0
pp_province_slug          0
pp_transaction_slug       0
scraped_timestamp         0
listing_date              0
description               0
rates_taxes               0
levies                    0
source_site               0
listing_id                0
bathrooms                 0
bedrooms                  0
property_type             0
province                  0
city                      0
suburb                    0
purchase_price            0
title                     0
listing_url               0
purchase_price_Rands      0
dtype: int64

## 9) Data Validation Checks

These prevent bad data from entering downstream systems.

In [42]:
assert (df_clean["purchase_price_Rands"] > 0).all(), "Negative prices found!"
assert df_clean["suburb"].notnull().all(), "Missing suburb values!"

print("✅ Validation checks passed")

✅ Validation checks passed


## 10) Final Dataset Summary

In [43]:
print("Final shape:", df_clean.shape)
df_clean.describe()

Final shape: (990, 26)


,purchase_price,bedrooms,bathrooms,parking_spaces,garage,floor_area_sqm,land_area_sqm,levies,rates_taxes,purchase_price_Rands
count,9.900000e+02,990.000000,990.000000,872.000000,422.000000,780.000000,572.000000,990.000000,990.000000,9.900000e+02
mean,2.630410e+07,2.688889,2.072222,2.850344,2.086493,166.882692,227.736014,2313.606815,1556.523129,2.630410e+06
std,3.748698e+07,1.463813,1.399330,2.514874,0.961359,184.502797,250.916154,1996.636768,1527.759230,3.748698e+06
min,1.500000e+06,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.500000e+05
25%,7.499238e+06,2.000000,1.000000,1.000000,1.000000,62.000000,4.000000,1130.000000,561.250000,7.499238e+05
50%,1.250000e+07,2.000000,2.000000,2.000000,2.000000,88.000000,181.500000,1900.000000,949.570946,1.250000e+06
75%,2.893000e+07,3.000000,2.000000,3.000000,2.000000,189.000000,317.000000,2833.250000,2000.000000,2.893000e+06
max,3.000000e+08,16.000000,12.000000,18.000000,6.000000,980.000000,996.000000,16000.000000,10828.000000,3.000000e+07


## 11) Exporting Clean Dataset

This dataset will be used for:
- EDA
- Feature engineering
- Model training

In [44]:
output_path = Path("data/processed/clean_sales_data.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)

print(f"✅ Clean data saved to: {output_path}")

✅ Clean data saved to: data\processed\clean_sales_data.csv
